In [1]:
# ================================
# Import libraries
# ================================
# Standard library
import glob
import json
import os
import shutil
from pathlib import Path

# Third-party library
import numpy as np
from PIL import Image
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_aws import BedrockEmbeddings
import boto3

print("✅ Environment ready")
print(os.getcwd())           # current directory
print(os.access(os.getcwd(), os.W_OK))  # is it writable?

✅ Environment ready
/Users/anirudh/Desktop/Courseera/IBM-RAG and agentic AI/RAG and Agentic AI Capstone Project
True


In [2]:
# ================================
# Verify vector database
# ================================

DB_DIR = str((Path.cwd() / "chroma_multimodal").resolve())

if not os.path.isdir(DB_DIR):
    raise RuntimeError(
        f"Vector database directory not found: '{DB_DIR}'. "
        "Please run Lesson 1 (Multimodal Vector Index Construction) first."
    )

article_db = Chroma(collection_name="restaurant_articles", persist_directory=DB_DIR)
image_db   = Chroma(collection_name="food_images",          persist_directory=DB_DIR)

n_articles = article_db._collection.count()
n_images   = image_db._collection.count()

if n_articles <= 0 or n_images <= 0:
    raise RuntimeError(
        "One or more collections are empty. Please rerun Lesson 1 to rebuild the index."
    )

print(f"✅ Article vectors: {n_articles}")
print(f"✅ Image vectors:   {n_images}")

/var/folders/kv/9b0l8lv56tsc3yvfy9l9nq180000gn/T/ipykernel_41510/1122168757.py:13: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  article_db = Chroma(collection_name="restaurant_articles", persist_directory=DB_DIR)


✅ Article vectors: 210
✅ Image vectors:   109


In [3]:
# ================================
# Initialize embedding models
# ================================
import base64
import numpy as np
from langchain_aws import BedrockEmbeddings
import boto3

# ---- Text embedding model (1024-d) ----
text_embedder = BedrockEmbeddings(
    model_id="amazon.titan-embed-text-v2:0",
    region_name="us-east-1",
    model_kwargs={"normalize": True}  # cosine-ready
)

def embed_texts(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        batch_embeddings = text_embedder.embed_documents(batch)
        embeddings.extend(batch_embeddings)
    return np.array(embeddings, dtype=np.float32)

print("✅ Text embedder ready")

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

def embed_images(paths, batch_size=16):
    embeddings = []
    for i in range(0, len(paths), batch_size):
        batch = paths[i:i+batch_size]
        for image_path in batch:
            # Encode image to base64
            with open(image_path, 'rb') as f:
                image_data = base64.b64encode(f.read()).decode('utf-8')

            # Call Titan Multimodal directly via boto3
            body = json.dumps({
                "inputImage": image_data,  # base64 encoded image
                "embeddingConfig": {
                    "outputEmbeddingLength": 1024
                }
            })

            response = bedrock_client.invoke_model(
                modelId="amazon.titan-embed-image-v1",
                body=body,
                contentType="application/json",
                accept="application/json"
            )

            result = json.loads(response['body'].read())
            embedding = result['embedding']

            # Normalize - cosine ready
            vector = np.array(embedding, dtype=np.float32)
            vector = vector / np.linalg.norm(vector)
            embeddings.append(vector)

    return np.array(embeddings, dtype=np.float32)

print("✅ Image embedder ready")

def embed_query_clip_text(query: str):
    # ✅ Use embed_texts instead of CLIP text encoder
    q_vec = embed_texts([query])[0]  # 1024-d, cosine-ready
    return q_vec

print("✅ Titan text embedder ready")


✅ Text embedder ready
✅ Image embedder ready
✅ Titan text embedder ready


In [4]:
# ================================
# Utilities
# ================================

def _unwrap(res: dict):
    """Chroma returns lists-of-lists; unwrap the first query."""
    ids   = res.get("ids", [[]])[0]
    docs  = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]
    return ids, docs, metas, dists

def _to_similarity(dists):
    """Convert 'smaller is better' distance to 'larger is better' similarity."""
    d = np.array(dists, dtype=np.float32)
    return 1.0 - d

def _minmax(x):
    """Min-max normalize to [0, 1] with safe handling for constant arrays."""
    x = np.array(x, dtype=np.float32)
    if x.size == 0:
        return x
    lo, hi = float(x.min()), float(x.max())
    if abs(hi - lo) < 1e-8:
        return np.ones_like(x)  # all equal -> treat as same confidence
    return (x - lo) / (hi - lo)

def print_hits(ids, docs, metas, scores, title: str, max_chars: int = 140):
    print(f"\n=== {title} ===")
    for i in range(len(ids)):
        meta = metas[i] if i < len(metas) else {}
        score = float(scores[i]) if i < len(scores) else None

        snippet = (docs[i] or "").replace("\n", " ").strip()
        if len(snippet) > max_chars:
            snippet = snippet[:max_chars].rstrip() + "..."

        cuisine = meta.get("cuisine", "N/A") if isinstance(meta, dict) else "N/A"
        location = meta.get("location", "N/A") if isinstance(meta, dict) else "N/A"
        doc_id = meta.get("doc_id", ids[i]) if isinstance(meta, dict) else ids[i]
        source = meta.get("source", "N/A") if isinstance(meta, dict) else "N/A"

        print(f"[{i+1}] id={doc_id} | cuisine={cuisine} | location={location} | source={source} | score={score:.4f}")
        print(f"{snippet}")

In [5]:
# ================================
# Retrieval functions
# ================================

def retrieve_articles(query: str, k: int = 5, where: dict | None = None):
    q_vec = embed_texts([query])[0]  # 384-d
    res = article_db._collection.query(
        query_embeddings=[q_vec.tolist()],
        n_results=k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    ids, docs, metas, dists = _unwrap(res)
    sims = _to_similarity(dists)
    return ids, docs, metas, sims

def retrieve_images_by_text(query: str, k: int = 5, where: dict | None = None):
    q_vec = embed_query_clip_text(query)  # 512-d
    res = image_db._collection.query(
        query_embeddings=[q_vec.tolist()],
        n_results=k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    ids, docs, metas, dists = _unwrap(res)
    sims = _to_similarity(dists)
    return ids, docs, metas, sims

print("✅ Retrieval functions ready")

✅ Retrieval functions ready


In [6]:
# ================================
# Multimodal fusion
# ================================

def fuse_rank(
    query: str,
    k_text: int = 5,
    k_img: int = 5,
    w_text: float = 0.6,
    w_img: float = 0.4,
    where_text: dict | None = None,
    where_img: dict | None = None,
    top_n: int = 5
):
    # Retrieve per modality
    t_ids, t_docs, t_metas, t_sims = retrieve_articles(query, k=k_text, where=where_text)
    i_ids, i_docs, i_metas, i_sims = retrieve_images_by_text(query, k=k_img, where=where_img)

    # Normalize within modality
    t_norm = _minmax(t_sims)
    i_norm = _minmax(i_sims)

    # Build one mixed candidate list with fused scores
    rows = []
    for j in range(len(t_ids)):
        rows.append({
            "modality": "article",
            "id": t_metas[j].get("doc_id", t_ids[j]) if isinstance(t_metas[j], dict) else t_ids[j],
            "cuisine": t_metas[j].get("cuisine", "N/A") if isinstance(t_metas[j], dict) else "N/A",
            "location": t_metas[j].get("location", "N/A") if isinstance(t_metas[j], dict) else "N/A",
            "source": t_metas[j].get("source", "N/A") if isinstance(t_metas[j], dict) else "N/A",
            "text_score": float(t_norm[j]),
            "img_score": 0.0,
            "fused": float(w_text * t_norm[j]),
            "snippet": (t_docs[j] or "").replace("\n", " ").strip(),
        })

    for j in range(len(i_ids)):
        rows.append({
            "modality": "image",
            "id": i_metas[j].get("doc_id", i_ids[j]) if isinstance(i_metas[j], dict) else i_ids[j],
            "cuisine": i_metas[j].get("cuisine", "N/A") if isinstance(i_metas[j], dict) else "N/A",
            "location": i_metas[j].get("location", "N/A") if isinstance(i_metas[j], dict) else "N/A",
            "source": i_metas[j].get("source", "N/A") if isinstance(i_metas[j], dict) else "N/A",
            "text_score": 0.0,
            "img_score": float(i_norm[j]),
            "fused": float(w_img * i_norm[j]),
            "snippet": (i_docs[j] or "").replace("\n", " ").strip(),
        })

    # Sort by fused score (desc rerank)
    rows.sort(key=lambda r: r["fused"], reverse=True)
    
    # if top_n not specified, return full pool (k_text + k_img)
    if top_n is None:
        return rows

    top_n = max(0, min(int(top_n), len(rows)))
    return rows[:top_n]

def print_fused(rows, title: str, max_chars: int = 90):
    print(f"\n=== {title} ===")
    for idx, r in enumerate(rows, start=1):
        snippet = r["snippet"]
        if len(snippet) > max_chars:
            snippet = snippet[:max_chars].rstrip() + "..."
        print(
            f"[{idx}] {r['modality']} | id={r['id']} | cuisine={r['cuisine']} | "
            f"location={r['location']} | fused={r['fused']:.4f} "
            f"(text={r['text_score']:.4f}, img={r['img_score']:.4f})"
        )
        print(snippet)

In [7]:
# ================================
# Demo 1 — Multimodal fusion (no filters)
# ================================

q = "cozy noodles with warm atmosphere"

rows = fuse_rank(
    q,
    k_text=5,
    k_img=5,
    w_text=0.6,
    w_img=0.4,
    where_text=None,
    where_img=None,
    top_n=5
)

print_fused(rows, title="Demo 1 — Multimodal fusion (no filters)")
print("✅ Demo 1 complete")


=== Demo 1 — Multimodal fusion (no filters) ===
[1] article | id=rest_73 | cuisine=Hand-Pulled Noodle | location=San Gabriel | fused=0.6000 (text=1.0000, img=0.0000)
Restaurant: The Noodle Nest Cuisine: Hand-Pulled Noodle Location: San Gabriel
[2] article | id=rest_49 | cuisine=instant noodle gourmet bowls, pineapple buns | location=Monterey Park | fused=0.4333 (text=0.7222, img=0.0000)
Restaurant: The Neon Noodle Bar Cuisine: instant noodle gourmet bowls, pineapple buns Loca...
[3] image | id=img_24 | cuisine=Japanese | location=N/A | fused=0.4000 (text=0.0000, img=1.0000)
Shoyu Ramen (Quick)
[4] image | id=img_55 | cuisine=Japanese | location=N/A | fused=0.1797 (text=0.0000, img=0.4493)
Okonomiyaki (Japanese Pancake)
[5] article | id=rest_150 | cuisine=Taiwanese | location=Irvine | fused=0.1193 (text=0.1988, img=0.0000)
Restaurant: The Silk Noodle Cuisine: Taiwanese Location: Irvine
✅ Demo 1 complete


In [8]:
# ================================
# Demo 2 — Multimodal fusion (metadata filters)
# ================================

q = "handmade pasta and romantic dinner"

where_articles = {"location": "pasadena"}
where_images = {"source": "recipe_image"}
rows = fuse_rank(
    q,
    k_text = 5, 
    k_img = 5, 
    w_text = 0.6, 
    w_img = 0.4, 
    where_text = where_articles, 
    where_img = where_images, 
    top_n = 5
)
if len(rows) == 0: 
    print("No results found.Please adjust your filter queries")
else:
    print_fused(rows, title = "Demo 2 — Multimodal fusion (metadata filters)")
print("Demo 2 is completed")    


=== Demo 2 — Multimodal fusion (metadata filters) ===
[1] image | id=img_32 | cuisine=Italian | location=N/A | fused=0.4000 (text=0.0000, img=1.0000)
Pesto Chicken Bake
[2] image | id=img_74 | cuisine=Thai | location=N/A | fused=0.2454 (text=0.0000, img=0.6135)
Thai Red Curry with Chicken
[3] image | id=img_24 | cuisine=Japanese | location=N/A | fused=0.2349 (text=0.0000, img=0.5873)
Shoyu Ramen (Quick)
[4] image | id=img_103 | cuisine=Chinese-American | location=N/A | fused=0.0220 (text=0.0000, img=0.0550)
General Tso's Chicken
[5] image | id=img_42 | cuisine=Indian | location=N/A | fused=0.0000 (text=0.0000, img=0.0000)
Garlic Naan
Demo 2 is completed


In [10]:
# ================================
# Demo 3 — Weight tuning
# ================================

q = "fresh sushi and minimalist presentation"

# TODO:
# 1. Run fusion ranking with a text-heavy setting and print results (use title="Demo 3A — Text-heavy fusion (w_text=0.8, w_img=0.2)")
# 2. Run fusion ranking with an image-heavy setting and print results (use title="Demo 3B — Image-heavy fusion (w_text=0.3, w_img=0.7)")
# 3. Select 5 results from each modality, but only show the top 5 fused results
text_heavy = fuse_rank(
    query = q, 
    k_text = 5,
    k_img = 5, 
    where_text = None, 
    where_img = None, 
    w_text = 0.8, 
    w_img = 0.2,
    top_n = 5
)
image_heavy = fuse_rank(
    query = q, 
    k_text = 5, 
    k_img = 5,
    where_text = None, 
    where_img = None, 
    w_text = 0.3, 
    w_img = 0.7,
    top_n = 5
)    
# your code here
print_fused(text_heavy, title="Demo 3A — Text-heavy fusion (w_text=0.8, w_img=0.2)")
print_fused(image_heavy,title="Demo 3B — Image-heavy fusion (w_text=0.3, w_img=0.7)" )
print("✅ Demo 3 complete")
print("🎉 Multimodal Similarity Fusion and Retrieval Ranking COMPLETE")



=== Demo 3A — Text-heavy fusion (w_text=0.8, w_img=0.2) ===
[1] article | id=rest_179 | cuisine=Japanese sushi | location=Carmel-by-the-Sea | fused=0.8000 (text=1.0000, img=0.0000)
Restaurant: The Silent Pebble Cuisine: Japanese sushi Location: Carmel-by-the-Sea
[2] article | id=rest_155 | cuisine=Sushi | location=Japantown (San Jose) | fused=0.5270 (text=0.6587, img=0.0000)
Restaurant: The Neon Koi Cuisine: Sushi Location: Japantown (San Jose)
[3] article | id=rest_185 | cuisine=Sushi | location=Japantown (San Jose) | fused=0.5270 (text=0.6587, img=0.0000)
Restaurant: The Neon Koi Cuisine: Sushi Location: Japantown (San Jose)
[4] image | id=img_17 | cuisine=Mediterranean | location=N/A | fused=0.2000 (text=0.0000, img=1.0000)
Shakshuka
[5] image | id=img_42 | cuisine=Indian | location=N/A | fused=0.1316 (text=0.0000, img=0.6581)
Garlic Naan

=== Demo 3B — Image-heavy fusion (w_text=0.3, w_img=0.7) ===
[1] image | id=img_17 | cuisine=Mediterranean | location=N/A | fused=0.7000 (text=0